In [1]:
import pandas as pd
import sys
import os
import urllib.request
!{sys.executable} -m pip install pyarrow

In [2]:
# this is the data we are going to work with right from the TLC website
# This will download the file from here: https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page
# this is 3,475,226 taxi rides! 
url = "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-01.parquet"
filename = "yellow_tripdata_2025-01.parquet"

if not os.path.exists(filename):
    print("Downloading trip data...")
    urllib.request.urlretrieve(url, filename)
    print("Done.")
else:
    print("File already exists, skipping download.")

File already exists, skipping download.


In [3]:
# We also need to download this "Taxi Zones Lookup Table" file
# which maps the LocationID number to a place name. It's small so we can
# just read it straight into a dataframe.
# https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv
zones = pd.read_csv("https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv")

In [4]:
zones.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 265 entries, 0 to 264
Data columns (total 4 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   LocationID    265 non-null    int64 
 1   Borough       264 non-null    object
 2   Zone          264 non-null    object
 3   service_zone  263 non-null    object
dtypes: int64(1), object(3)
memory usage: 8.4+ KB


In [5]:
# Parquet is a file format for storing tabular data (like a spreadsheet or database table) 
# that's optimized for fast reading and small file sizes. You will often find large datasets 
# distrubuted in this format. Easy for Python to read into a plain old dataframe!
yellow_cabs = pd.read_parquet('yellow_tripdata_2025-01.parquet')

In [6]:
yellow_cabs.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3475226 entries, 0 to 3475225
Data columns (total 20 columns):
 #   Column                 Dtype         
---  ------                 -----         
 0   VendorID               int32         
 1   tpep_pickup_datetime   datetime64[us]
 2   tpep_dropoff_datetime  datetime64[us]
 3   passenger_count        float64       
 4   trip_distance          float64       
 5   RatecodeID             float64       
 6   store_and_fwd_flag     object        
 7   PULocationID           int32         
 8   DOLocationID           int32         
 9   payment_type           int64         
 10  fare_amount            float64       
 11  extra                  float64       
 12  mta_tax                float64       
 13  tip_amount             float64       
 14  tolls_amount           float64       
 15  improvement_surcharge  float64       
 16  total_amount           float64       
 17  congestion_surcharge   float64       
 18  Airport_fee           

In [9]:
# We need to do lookup for PULocationID and DULocationID so we can read them.
# this will make a 'dictionary' for us to use to map the numbers to new columns with the names
zone_lookup = zones.set_index('LocationID')['Zone'].to_dict()

In [11]:
# this will make a new column named PU_Zone that looks up the PULocationID number from the zones
yellow_cabs['PU_Zone'] = yellow_cabs['PULocationID'].map(zone_lookup)

# this will make a new column named DU_Zone that looks up the PULocationID number from the zones
yellow_cabs['DO_Zone'] = yellow_cabs['DOLocationID'].map(zone_lookup)

In [14]:
yellow_cabs.head()

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,...,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee,PU_Zone,DO_Zone
0,1,2025-01-01 00:18:38,2025-01-01 00:26:59,1.0,1.60,1.0,N,229,237,1,...,0.5,3.00,0.0,1.0,18.00,2.5,0.0,0.0,Sutton Place/Turtle Bay North,Upper East Side South
1,1,2025-01-01 00:32:40,2025-01-01 00:35:13,1.0,0.50,1.0,N,236,237,1,...,0.5,2.02,0.0,1.0,12.12,2.5,0.0,0.0,Upper East Side North,Upper East Side South
2,1,2025-01-01 00:44:04,2025-01-01 00:46:01,1.0,0.60,1.0,N,141,141,1,...,0.5,2.00,0.0,1.0,12.10,2.5,0.0,0.0,Lenox Hill West,Lenox Hill West
3,2,2025-01-01 00:14:27,2025-01-01 00:20:01,3.0,0.52,1.0,N,244,244,2,...,0.5,0.00,0.0,1.0,9.70,0.0,0.0,0.0,Washington Heights South,Washington Heights South
4,2,2025-01-01 00:21:34,2025-01-01 00:25:06,3.0,0.66,1.0,N,244,116,2,...,0.5,0.00,0.0,1.0,8.30,0.0,0.0,0.0,Washington Heights South,Hamilton Heights
